# MalImg Family Classification

**Task 3 — Machine Learning with Image Data | Semester I, 2026–2027**

Base để nhóm review, chỉnh cấu hình và chạy thực nghiệm. Dataset MalImg 24 lớp
(loại Yuner.A) đang chờ giảng viên xác nhận. Notebook chưa có kết quả benchmark
hoặc bằng chứng đã chạy `Run all` từ Colab sạch.

- A: dữ liệu, EDA, split; B: đặc trưng, classifier; C: đánh giá, tích hợp.
- Điền tên nhóm, họ tên/MSSV/email, mã môn và giảng viên trước khi nộp.
- Repo: https://github.com/TuanKhai1210/malimg-family-classification
- Xem `docs/review_and_train.md` và `reports/review_checklist.md`.

Trong Colab chọn **Runtime → Change runtime type → GPU** trước khi chạy pretrained
features. Không cần mount Drive. `prepare` dành cho review; `all` chạy EDA,
trích/lưu/nạp features và train/chọn C bằng validation. Chỉ bật final test sau
khi nhóm khóa phương án. Bản nộp cuối cần chạy toàn bộ trên runtime sạch và có report.


## 1. Chọn chế độ

Chỉnh một cell dưới đây, rồi **Runtime → Run all**. Lần đầu giữ `STAGE='prepare'`
để kiểm kê dữ liệu, split và EDA. Sau review đổi sang `STAGE='all'` để train.
`FINAL_TEST=True` mở đánh giá test sau chọn bằng validation; kết quả được khóa,
không tự chạy lại khi thay cấu hình. Dữ liệu ~1.17 GB, weights sẽ tải từ PyTorch.


In [1]:
STAGE = 'prepare'  # prepare | eda | extract | train | final | all
FINAL_TEST = False  # dùng với all sau khi khóa protocol; final là bước riêng
INSTALL_DEPENDENCIES = True
REPO_URL = 'https://github.com/TuanKhai1210/malimg-family-classification.git'
REPO_REF = 'feat/resumable-drive-checkpoints'
CONFIG_FILE = 'configs/project.json'
ARCHIVE_PATH = None  # Colab tự tải link công khai; local có thể dùng ZIP đã kiểm

# Sao lưu cá nhân tùy chọn; Run all mặc định không cần Drive.
USE_DRIVE_BACKUP = False  # đổi True để mount Drive và tự sao lưu sau mỗi trial
BACKUP_DIR = '/content/drive/MyDrive/ML_MalImg_Backups'
RESUME_RUN = None  # ví dụ 'validation_20261023T...'; không tự chọn run
RESTORE_SNAPSHOT = None  # đường dẫn snapshots/...zip trên Drive; dùng trong runtime mới
RESTORE_TO = '/content/malimg-restored'  # phải là thư mục chưa tồn tại
WORKING_ROOT = None  # sau restore/restart: đặt đường dẫn workspace đã phục hồi


## 2. Chuẩn bị code và thư viện

Cell này dùng repo hiện tại nếu đã có, hoặc clone đúng nhánh review. Ghi lại commit
và môi trường thật cho mỗi lần chạy. Giữ sẵn PyTorch của Colab nếu đáp ứng yêu cầu;
nếu cài/thay thư viện trong một runtime đã dùng trước đó, hãy restart và Run all.

Google Drive chỉ là bản sao cá nhân: đặt `USE_DRIVE_BACKUP=True` khi thực nghiệm.
Code và dữ liệu vẫn chạy từ ổ đĩa Colab, nguồn dữ liệu vẫn công khai. Khi mất phiên,
chọn `RESTORE_SNAPSHOT` và `RESUME_RUN` rõ ràng rồi chạy setup trong runtime mới.
Setup khôi phục đúng source trong backup và pin numpy/scipy/sklearn/joblib đã lưu;
Python, PyTorch/Torchvision và loại thiết bị cũng cần khớp để reuse feature cache.
Xem [hướng dẫn lưu/khôi phục](../docs/checkpoint_and_recovery.md).


In [2]:
from pathlib import Path
import json
import subprocess
import sys

if USE_DRIVE_BACKUP:
    from google.colab import drive
    drive.mount('/content/drive')  # bạn tự chọn tài khoản và duyệt quyền

# Working folder riêng để không sửa checkout/các artifacts của phiên cũ.
candidates = ([Path(WORKING_ROOT)] if WORKING_ROOT else
              [Path.cwd(), Path.cwd() / 'malimg-family-classification-checkpoints',
               Path('/content/malimg-family-classification-checkpoints')])
repo_root = next((path for path in candidates
                  if (path / 'scripts/manage_checkpoints.py').is_file()), None)
if repo_root is None and WORKING_ROOT:
    raise FileNotFoundError('WORKING_ROOT does not contain the checkpoint source; inspect Files.')
if repo_root is None:
    parent = Path('/content') if Path('/content').is_dir() else Path.cwd()
    repo_root = parent / 'malimg-family-classification-checkpoints'
    if repo_root.exists():
        raise RuntimeError('Working folder exists but lacks checkpoint tools; inspect it before continuing.')
    subprocess.run(['git', 'clone', '--branch', REPO_REF, '--depth', '1', REPO_URL, str(repo_root)], check=True)

if RESTORE_SNAPSHOT:
    # Subprocess avoids importing bootstrap modules before the restored source.
    subprocess.run([sys.executable, str(repo_root / 'scripts/manage_checkpoints.py'),
                    'restore', RESTORE_SNAPSHOT, '--destination', RESTORE_TO], check=True)
    repo_root = Path(RESTORE_TO).resolve()

restored = repo_root / 'RESTORED_BACKUP.json'
restore_record = json.loads(restored.read_text(encoding='utf-8')) if restored.is_file() else {}
if restore_record and restore_record['python'].split()[0].split('.')[:2] != list(map(str, sys.version_info[:2])):
    raise RuntimeError('Python major/minor differs from backup; choose a matching runtime before resume.')
if INSTALL_DEPENDENCIES:
    pins = [name + '==' + version for name, version in restore_record.get('software', {}).items()
            if name in {'numpy', 'scipy', 'scikit-learn', 'joblib'}]
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
                    str(repo_root / 'requirements.txt'), *pins], check=True)
loaded = sys.modules.get('modules.pipeline')
if loaded and not Path(loaded.__file__).resolve().is_relative_to(repo_root.resolve()):
    raise RuntimeError('Another source is already imported. Restart the Python session, then run setup in order.')
sys.path.insert(0, str(repo_root))
print('Repository:', repo_root)
if (repo_root / '.git').exists():
    print('Commit:', subprocess.check_output(['git', '-C', str(repo_root), 'rev-parse', 'HEAD'], text=True).strip())
elif (repo_root / 'PACKAGE.json').is_file():
    print('Packaged source commit:', json.loads((repo_root / 'PACKAGE.json').read_text(encoding='utf-8')).get('source_commit'))
subprocess.run([sys.executable, str(repo_root / 'scripts/check_environment.py')], check=True)
print('Backup directory:', BACKUP_DIR if USE_DRIVE_BACKUP else 'disabled')
print('Validation runs:', [path.name for path in sorted((repo_root / 'results/runs').glob('validation_*'))])
if USE_DRIVE_BACKUP:
    print('Recent snapshots:', [str(path) for path in sorted((Path(BACKUP_DIR) / 'snapshots').glob('*.zip'))[-5:]])


Repository: /content/malimg-family-classification-checkpoints
Commit: 582f249714ef2e6e4d334dcda86ea0d9c80e2bcb
Backup directory: disabled
Validation runs: []


## 3. Cấu hình chung

Core: E2 ResNet50+LR, E3 ResNet50+SVM, E4 ConvNeXt-Tiny+SVM. Chỉ train được fit
scaler/PCA. C được chọn bằng Macro-F1 của đại diện nhóm pixel trên validation.
Tất cả trial, cảnh báo hội tụ, config, sample IDs và provenance được lưu.

Nguồn ảnh công khai (download trực tiếp, không mount cloud drive):
https://www.dropbox.com/scl/fi/wdb6omeiu2lg796qvt9l7/malimg_dataset.zip?rlkey=63q2xqmtlm66gilf6idd2c9k7&dl=1

Checksum bản đã kiểm: `9766ae9f1daa520e367fb486ca94728fe1485c0f5cb8314c312d77089a1fe9ec`.


In [3]:
from modules.config import load_config, validate_config

config = load_config(repo_root / CONFIG_FILE)
if RESUME_RUN:
    if restore_record:
        config = restore_record['effective_config']
    run_config = repo_root / config.get('results_dir', 'results') / 'runs' / RESUME_RUN / 'config.json'
    config = load_config(run_config)  # dùng đúng cấu hình của run đã chọn
# Chỉnh các lựa chọn tại đây trước lần validation đầu tiên:
# config['features']['batch_size'] = 16
# config['features']['device'] = 'cuda'
# config['features']['resize_policy'] = 'pad'
# config['training']['pca_components'] = 0.95
validate_config(config)
print(json.dumps(config, ensure_ascii=False, indent=2))


{
  "dataset_name": "MalImg",
  "dataset_url": "https://www.dropbox.com/scl/fi/wdb6omeiu2lg796qvt9l7/malimg_dataset.zip?rlkey=63q2xqmtlm66gilf6idd2c9k7&dl=1",
  "archive_bytes": 1174609734,
  "archive_sha256": "9766ae9f1daa520e367fb486ca94728fe1485c0f5cb8314c312d77089a1fe9ec",
  "excluded_families": [
    "Yuner.A"
  ],
  "split_seed": 42,
  "split_target": {
    "train": 0.7,
    "val": 0.15,
    "test": 0.15
  },
  "primary_metric": "macro_f1_unique_pixel_representatives",
  "manifest_version": "v1",
  "instructor_approved": false,
  "status": "reviewable implementation; instructor approval and benchmark runs pending",
  "data_dir": "data/images",
  "results_dir": "results",
  "features": {
    "image_size": 224,
    "resize_policy": "direct",
    "batch_size": 32,
    "device": "auto",
    "num_workers": 0,
    "cache": true,
    "weights": {
      "resnet50": "IMAGENET1K_V2",
      "convnext_tiny": "IMAGENET1K_V1"
    }
  },
  "training": {
    "seed": 42,
    "max_iter": 5000,
   

## 4. Chạy pipeline

Prepare kiểm archive, chỉ giải nén ảnh, tạo manifest/label map/split theo nhóm pixel,
kiểm leakage và dựng EDA. Feature cache chỉ tái dùng khi provenance và hashes khớp.
Test features có thể được trích sẵn; validation không nạp mảng test hoặc chấm test.

`all` giữ test kín khi `FINAL_TEST=False`. Sau khi nhóm khóa phương án,
chọn `STAGE='final'` trong cùng runtime, hoặc `all` + `FINAL_TEST=True` cho lần
kiểm tra bản nộp từ runtime sạch. Không bỏ lock để tìm cấu hình theo test.

Mỗi trial có `trial.json` trạng thái running/completed/interrupted, log thời gian CPU
mỗi 30 giây và checkpoint sau khi lưu đủ model/kết quả. Với Drive bật, features
được đóng bundle dùng chung, còn mỗi mốc tạo snapshot ZIP mới giữ mọi runs đã lưu.
`RESUME_RUN` chỉ bỏ qua trial hoàn tất đã kiểm hash; fit đang dở phải chạy lại.
Không resume run của code cũ thiếu resume.json hoặc code/config/feature khác.


In [ ]:
if 'repo_root' not in globals() or not (repo_root / 'modules/pipeline.py').is_file():
    raise RuntimeError('Chạy cell 1 và 2 setup trước; Python hiện chưa có repo_root hợp lệ.')
sys.path.insert(0, str(repo_root))
from modules.pipeline import run_pipeline

if FINAL_TEST and STAGE != 'all':
    raise ValueError('Use FINAL_TEST with all, or choose the explicit final stage.')
run = run_pipeline(repo_root, config, stage=STAGE,
                   archive_path=ARCHIVE_PATH, final_test=FINAL_TEST,
                   resume_run=RESUME_RUN,
                   backup_dir=BACKUP_DIR if USE_DRIVE_BACKUP else None)
print(json.dumps(run, ensure_ascii=False, indent=2, default=str))


## 5. Xem EDA và kết quả thật

Hai đơn vị đánh giá được trình bày riêng: mỗi nhóm pixel một đại diện và toàn bộ
file. Macro-F1 là metric chính. Các bảng sau chỉ hiện khi đã có output thật;
không dùng kết quả synthetic smoke tests trong báo cáo MalImg.


In [ ]:
import pandas as pd
from IPython.display import display, Image as DisplayImage

results_root = repo_root / config.get('results_dir', 'results')
summary_path = Path(run['dataset']['summary_path'])
print(summary_path.read_text(encoding='utf-8'))
for path in sorted((results_root / 'eda').glob('*.png')):
    print(path.name)
    display(DisplayImage(filename=str(path)))
comparison = results_root / 'comparison.csv'
if comparison.is_file():
    display(pd.read_csv(comparison))
else:
    print('Chưa chạy classifier; chưa có bảng benchmark.')


## 6. Sao lưu và đóng gói

Bật Drive từ cell 1 để tự sao lưu sau mỗi extractor, trial, validation hoàn tất
và khi nhận KeyboardInterrupt. Mất VM đột ngột chỉ còn mốc đã sao lưu thành công.
Giữ cả features/ và snapshots/ trên Drive; snapshot ZIP cần feature bundle đi kèm.
Không tự xóa lịch sử. Chỉ restore backups do nhóm tạo vì model joblib là mã tuần tự hóa.
Draft ZIP dưới đây là lựa chọn tải về máy, chưa phải gói nộp hoàn chỉnh.
Xem docs/checkpoint_and_recovery.md cho cách cứu phiên cũ và lỗi import modules.


In [ ]:
CREATE_DRAFT_ZIP = False  # đổi True khi muốn tải code và artifacts về máy
if CREATE_DRAFT_ZIP:
    from modules.submission import build_submission
    package = build_submission(repo_root, repo_root / 'dist/malimg_review_base.zip', draft=True)
    print('Draft ZIP:', package)
    try:
        from google.colab import files
    except ImportError:
        print('Local runtime: mở file ZIP ở đường dẫn trên.')
    else:
        files.download(str(package))


## 7. Checklist và tham khảo

- Review dữ liệu, chính sách Yuner.A và split; xác nhận phạm vi với giảng viên.
- Kiểm feature/image/label alignment; chốt weights, resize, C grid và train unit.
- Chạy validation, đọc warnings; khóa lựa chọn rồi đánh giá test.
- Điền báo cáo từ artifacts thật và ghi nhận phần nhóm đã review/chỉnh sửa.
- Thành viên thứ hai kiểm notebook từ Colab sạch; lưu log/evidence.
- Xuất `reports/final_report.pdf`, hoàn thiện README/thành viên và tạo ZIP cuối.

Nataraj et al., [Malware Images: Visualization and Automatic Classification](https://doi.org/10.1145/2016904.2016908), VizSec 2011.
[Torchvision ResNet50](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet50.html),
[ConvNeXt-Tiny](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.convnext_tiny.html),
[scikit-learn model persistence](https://scikit-learn.org/stable/model_persistence.html).
